# 02 — PCA experiments

E0 raw-pixel baseline, E1 PCA + 3 classifiers, E3 eigenspectrum

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

from src.classifiers import min_euclid_predict, min_mahal_predict, nn_predict
from src.config import OFFSETS
from src.data import load_coil20, make_split
from src.reduction import fit_pca, pca_project

In [ ]:
X, y, pose = load_coil20()

## PCA sanity checks

Run on every split: eigenvalues sorted, eigenvectors orthonormal, N-th eigenvalue ~0, match with sklearn (cross-check only).

In [ ]:
from sklearn.decomposition import PCA  # cross-check only

for off in OFFSETS:
    tr, _ = make_split(pose, off)
    N = len(tr)
    m, V, lam = fit_pca(X[tr])
    assert V.shape == (X.shape[1], N - 1) and lam.shape == (N - 1,)
    assert np.all(np.diff(lam) <= 0)
    assert np.allclose(V.T @ V, np.eye(N - 1), atol=1e-10)
    assert np.linalg.svd(X[tr] - m, compute_uv=False)[-1] ** 2 / N < 1e-20   # dropped component
    assert np.allclose(lam * N / (N - 1), PCA().fit(X[tr]).explained_variance_[:N - 1])
    Z = pca_project(X[tr], m, V, N - 1)
    assert np.allclose(Z.mean(axis=0), 0) and np.allclose(Z.var(axis=0), lam)
print("PCA checks passed")

## Classifier sanity checks

Toy case: both classes spread widely in x and narrowly in y, so the query is Euclidean-closer to class 0's mean but Mahalanobis-closer to class 1. Then agreement with reference implementations on split 0 (cross-check only).

In [ ]:
Ztoy = np.array([[-6, 0.1], [6, -0.1], [-6, -0.1], [6, 0.1],      # class 0, mean (0, 0)
                 [-2, 1.1], [10, 0.9], [-2, 0.9], [10, 1.1]])     # class 1, mean (4, 1)
ytoy = np.array([0, 0, 0, 0, 1, 1, 1, 1])
q = np.array([[1.8, 0.95]])
assert min_euclid_predict(Ztoy, ytoy, q)[0] == 0
assert min_mahal_predict(Ztoy, ytoy, q)[0] == 1
assert (nn_predict(Ztoy, ytoy, Ztoy) == ytoy).all()   # each training point is its own nearest neighbour
print("toy checks passed")

In [ ]:
from scipy.spatial.distance import cdist                # cross-check only
from sklearn.neighbors import KNeighborsClassifier      # cross-check only

tr, te = make_split(pose, 0)
m, V, lam = fit_pca(X[tr])
N, c = len(tr), len(np.unique(y[tr]))
for d in (5, 50, N - c):
    Ztr, Zte = pca_project(X[tr], m, V, d), pca_project(X[te], m, V, d)
    M = np.stack([Ztr[y[tr] == k].mean(axis=0) for k in range(c)])
    R = Ztr - M[y[tr]]
    VI = np.linalg.inv(R.T @ R / (N - c))
    assert (nn_predict(Ztr, y[tr], Zte) == KNeighborsClassifier(n_neighbors=1).fit(Ztr, y[tr]).predict(Zte)).all()
    assert (min_euclid_predict(Ztr, y[tr], Zte) == cdist(Zte, M).argmin(axis=1)).all()
    assert (min_mahal_predict(Ztr, y[tr], Zte) == cdist(Zte, M, "mahalanobis", VI=VI).argmin(axis=1)).all()
    print(f"d={d}: " + ", ".join(f"{f.__name__} {np.mean(f(Ztr, y[tr], Zte) == y[te]):.3f}"
                                for f in (nn_predict, min_euclid_predict, min_mahal_predict)))

Ztr, Zte = pca_project(X[tr], m, V, N - c + 1), pca_project(X[te], m, V, N - c + 1)
try:
    min_mahal_predict(Ztr, y[tr], Zte)
    raise AssertionError("expected ValueError above the cap")
except ValueError as e:
    print("cap guard:", e)